In [ ]:
%cd /home/smalani/PartialObservations_URP

import numpy as np
import matplotlib.pyplot as plt
from URPModel import URP_metrics

In [ ]:
from config import config
config["MODEL"]["BOX"] = 'Grey'
config["MODEL"]["Parameters"] = 'Fixed'

filename = '/home/smalani/PartialObservations_URP/data/Case8a/' + "model_run_" + "0" + ".net"
network = URP_metrics.load_network(filename)

In [ ]:
import torch

def true_ode(t, x, Da, B, beta):
    x1, x2 = x
    dx1dt = -x1 + Da * (1-x1) * np.exp(x2)
    dx2dt = -x2 + B * Da * (1-x1) * np.exp(x2) - beta * x2
    return np.array([dx1dt, dx2dt])

def pred_ode(t, x, Da, B, beta):
    x_input = torch.from_numpy(x).to(network.device)
    if x_input.ndim > 1:
        par_input = (torch.zeros((x_input.shape[1],1)) + Da).to(network.device)
    else:
        par_input = (torch.zeros((1,)) + Da).to(network.device)
    # par_input = torch.tensor([Da]).to(network.device)
    # while par_input.ndim < x_input.ndim:
    #     par_input = par_input.unsqueeze(0)
    g = network.raw_output(x_input.T, par_input).detach().numpy()

    x1, x2 = x
    dx1dt = -x1 + g
    dx2dt = -x2 + B * g - beta * x2
    return [dx1dt, dx2dt]


In [ ]:
from scipy.integrate import solve_ivp

xin = np.array([0.82, 2.5])
Da = 0.3
B = 14.8
beta = 3
t_span = [0, 100]
t_eval = np.linspace(0, t_span[-1], 10001)

sol_true = solve_ivp(true_ode, t_span, xin, t_eval=t_eval, args=(Da, B, beta))
sol_pred = solve_ivp(pred_ode, t_span, xin, t_eval=t_eval, args=(Da, B, beta))

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(211)
ax.plot(sol_true.t, sol_true.y[0], label='$x_{1,true}$', linewidth=2)
ax.plot(sol_pred.t, sol_pred.y[0], label='$x_{1,pred}$', linewidth=2)
ax.set_xlabel('$t$', fontsize=16)
ax.set_ylabel('$x_{1}$', fontsize=16)
ax.legend()

ax = fig.add_subplot(212)
ax.plot(sol_true.t, sol_true.y[1], label='$x_{2,true}$', linewidth=2)
ax.plot(sol_pred.t, sol_pred.y[1], label='$x_{2,pred}$', linewidth=2)
ax.set_xlabel('$t$', fontsize=16)
ax.set_ylabel('$x_{2}$', fontsize=16)
ax.legend()

In [ ]:
from scipy.integrate import solve_ivp

xin = np.array([0.82, 2.5])
Da = 0.3
# B = 11
beta = 3
t_span = [0, 20]
t_eval = np.linspace(0, 20, 201)

B_arr = np.linspace(7, 15, 41)
B_err_arr = np.zeros((2, B_arr.size, t_eval.size))

for i, B in enumerate(B_arr):
    sol_true = solve_ivp(true_ode, t_span, xin, t_eval=t_eval, args=(Da, B, beta))
    sol_pred = solve_ivp(pred_ode, t_span, xin, t_eval=t_eval, args=(Da, B, beta))

    B_err_arr[:, i, :] = np.abs(sol_true.y - sol_pred.y)

X,Y = np.meshgrid(B_arr, t_eval)

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(211)
s = ax.pcolor(X, Y, B_err_arr[0, :, :].T, cmap='jet')
ax.set_xlabel('$B$', fontsize=16)
ax.set_ylabel('$t$', fontsize=16)
ax.set_title('$x_{1}$', fontsize=16)
ax.vlines([11], 0, 20, colors='red', linewidth=4, linestyles='dashed')
plt.colorbar(s)

ax = fig.add_subplot(212)
s = ax.pcolor(X, Y, B_err_arr[1, :, :].T, cmap='jet')
ax.set_xlabel('$B$', fontsize=16)
ax.set_ylabel('$t$', fontsize=16)
ax.set_title('$x_{2}$', fontsize=16)
ax.vlines([11], 0, 20, colors='red', linewidth=4, linestyles='dashed')
plt.colorbar(s)
plt.tight_layout()

In [ ]:
from scipy.integrate import solve_ivp

xin = np.array([0.82, 2.5])
Da = 0.3
# B = 11
beta = 3
t_span = [0, 20]
t_eval = np.linspace(0, 20, 201)

B_arr = np.linspace(7, 15, 101)
B_err_arr = np.zeros((2, B_arr.size, t_eval.size))

for i, B in enumerate(B_arr):
    sol_true = solve_ivp(true_ode, t_span, xin, t_eval=t_eval, args=(Da, B, beta))

    RHS_true = true_ode(0, sol_true.y, Da, B, beta)
    RHS_pred = pred_ode(0, sol_true.y, Da, B, beta)
    # sol_pred = solve_ivp(pred_ode, t_span, xin, t_eval=t_eval, args=(Da, B, beta))

    # B_err_arr[:, i, :] = np.abs(sol_true.y - sol_pred.y)
    B_err_arr[:, i, :] = np.abs(RHS_true - RHS_pred)

X,Y = np.meshgrid(B_arr, t_eval)

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(211)
s = ax.pcolor(X, Y, B_err_arr[0, :, :].T, cmap='jet')
ax.set_xlabel('$B$', fontsize=16)
ax.set_ylabel('$t$', fontsize=16)
ax.set_title('$\\dot{x_{1}}$', fontsize=16)
ax.vlines([11], 0, 20, colors='red', linewidth=4, linestyles='dashed')
plt.colorbar(s)

ax = fig.add_subplot(212)
s = ax.pcolor(X, Y, B_err_arr[1, :, :].T, cmap='jet')
ax.set_xlabel('$B$', fontsize=16)
ax.set_ylabel('$t$', fontsize=16)
ax.set_title('$\\dot{x_{2}}$', fontsize=16)
ax.vlines([11], 0, 20, colors='red', linewidth=4, linestyles='dashed')
plt.colorbar(s)
plt.tight_layout()

In [ ]:
from scipy.integrate import solve_ivp

xin = np.array([0.82, 2.5])
Da = 0.3
B = 11
# beta = 3
t_span = [0, 20]
t_eval = np.linspace(0, 20, 201)

beta_arr = np.linspace(1, 5, 101)
beta_err_arr = np.zeros((2, B_arr.size, t_eval.size))

for i, beta in enumerate(beta_arr):
    # sol_true = solve_ivp(true_ode, t_span, xin, t_eval=t_eval, args=(Da, B, beta))
    # sol_pred = solve_ivp(pred_ode, t_span, xin, t_eval=t_eval, args=(Da, B, beta))

    # beta_err_arr[:, i, :] = np.abs(sol_true.y - sol_pred.y)

    sol_true = solve_ivp(true_ode, t_span, xin, t_eval=t_eval, args=(Da, B, beta))

    RHS_true = true_ode(0, sol_true.y, Da, B, beta)
    RHS_pred = pred_ode(0, sol_true.y, Da, B, beta)
    # sol_pred = solve_ivp(pred_ode, t_span, xin, t_eval=t_eval, args=(Da, B, beta))

    # B_err_arr[:, i, :] = np.abs(sol_true.y - sol_pred.y)
    beta_err_arr[:, i, :] = np.abs(RHS_true - RHS_pred)

X,Y = np.meshgrid(beta_arr, t_eval)

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(211)
s = ax.pcolor(X, Y, beta_err_arr[0, :, :].T, cmap='jet')
ax.set_xlabel('$\\beta$', fontsize=16)
ax.set_ylabel('$t$', fontsize=16)
ax.set_title('$\dot{x_{1}}$', fontsize=16)
ax.vlines([3], 0, 20, colors='red', linewidth=4, linestyles='dashed')
plt.colorbar(s)

ax = fig.add_subplot(212)
s = ax.pcolor(X, Y, beta_err_arr[1, :, :].T, cmap='jet')
ax.set_xlabel('$\\beta$', fontsize=16)
ax.set_ylabel('$t$', fontsize=16)
ax.set_title('$\\dot{x_{2}}$', fontsize=16)
ax.vlines([3], 0, 20, colors='red', linewidth=4, linestyles='dashed')
plt.colorbar(s)
plt.tight_layout()

In [ ]:
from scipy.integrate import solve_ivp

xin = np.array([0.82, 2.5])
Da = 0.3
B = 11
# beta = 3
t_span = [0, 20]
t_eval = np.linspace(0, 20, 201)

beta_arr = np.linspace(1, 5, 41)
beta_err_arr = np.zeros((2, B_arr.size, t_eval.size))

for i, beta in enumerate(beta_arr):
    sol_true = solve_ivp(true_ode, t_span, xin, t_eval=t_eval, args=(Da, B, beta))
    sol_pred = solve_ivp(pred_ode, t_span, xin, t_eval=t_eval, args=(Da, B, beta))

    beta_err_arr[:, i, :] = np.abs(sol_true.y - sol_pred.y)

X,Y = np.meshgrid(beta_arr, t_eval)

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(211)
s = ax.pcolor(X, Y, beta_err_arr[0, :, :].T, cmap='jet')
ax.set_xlabel('$\\beta$', fontsize=16)
ax.set_ylabel('$t$', fontsize=16)
ax.set_title('$x_{1}$', fontsize=16)
ax.vlines([3], 0, 20, colors='red', linewidth=4, linestyles='dashed')
plt.colorbar(s)

ax = fig.add_subplot(212)
s = ax.pcolor(X, Y, beta_err_arr[1, :, :].T, cmap='jet')
ax.set_xlabel('$\\beta$', fontsize=16)
ax.set_ylabel('$t$', fontsize=16)
ax.set_title('$x_{2}$', fontsize=16)
ax.vlines([3], 0, 20, colors='red', linewidth=4, linestyles='dashed')
plt.colorbar(s)
plt.tight_layout()

In [ ]:
from scipy.optimize import fsolve

def ODE_Bifurc(y, func, Da, x10):
    x2, T = y
    # pars = get_pars(Da)
    
#     event = ODE_Event
#     event.terminal = True
    
    y0 = [x10, x2]

    
    sol = solve_ivp(func, y0=y0, t_span=[0, 0.1],
#                     args=pars,
                    rtol=1e-5, atol=1e-8, dense_output=True)#, events=(event,))#, dense_output=True)
    
    y_init = sol.y[:,-1]
    
    sol = solve_ivp(func, y0=y_init, t_span=[0.1, T],
#                     args=pars,
                    rtol=1e-5, atol=1e-8, dense_output=True)#, events=(event,))#, dense_output=True)
    
    T_out = sol.t[-1]
    x1_out = sol.y[0,-1]
    x2_out = sol.y[1,-1]

    return (x10-x1_out), (x2-x2_out)

##############################################################################################33

B_arr = np.linspace(10,15,100)
Da = 0.3

stable_ss = []
stable_B = []

unstable_ss = []
unstable_B = []

x1max_pred = []
x1min_pred = []

x2max_pred = []
x2min_pred = []

roots = []
switch = False
root = [0.5, 1]
for i in range(len(B_arr)):
    print(i)
    B = B_arr[i]
    beta = 3
    
    # pars = get_pars(Da)

    def torch_function(x):
        g = network.raw_output(x.unsqueeze(0).to(network.device),
                                                torch.tensor([Da]).unsqueeze(0).to(network.device))
        x1, x2 = x
        dx1dt = -x1 + g
        dx2dt = -x2 + B * g - beta * x2
        return torch.cat((dx1dt, dx2dt), dim=-1)

    def numpy_function(x):
        g = network.raw_output(torch.tensor(x).unsqueeze(0).to(network.device),
                                                   torch.tensor([Da]).unsqueeze(0).to(network.device))
        x1, x2 = x
        dx1dt = -x1 + g
        dx2dt = -x2 + B * g - beta * x2
        return torch.cat((dx1dt, dx2dt), dim=-1).detach().cpu().squeeze().numpy()

    def numpy_function_integ(t, x):
        g = network.raw_output(torch.tensor(x).unsqueeze(0).to(network.device),
                                                   torch.tensor([Da]).unsqueeze(0).to(network.device))
        x1, x2 = x
        dx1dt = -x1 + g
        dx2dt = -x2 + B * g - beta * x2
        return torch.cat((dx1dt, dx2dt), dim=-1).detach().cpu().squeeze().numpy()

    
    root = fsolve(numpy_function, root)
    J = torch.autograd.functional.jacobian(torch_function, torch.from_numpy(root), strict=True)
    w, v = np.linalg.eig(J)

    if np.any(w.real>0):
        if switch is not True:
            switch = True
            stable_ss.append(np.array([np.nan, np.nan]))
            stable_B.append(np.nan)
            
        unstable_ss.append(root)
        unstable_B.append(B)

        solved = False
        perturb = 0.1
        
        y0 = [root[1], 2]
        while not solved:
            x10 = root[0] + perturb
            

            y, infodict, ier, mesg = fsolve(ODE_Bifurc, y0, args=(numpy_function_integ, Da, x10), full_output=True)
            
            if ier == 1:
                solved = True
                y0=y
            else:
                perturb = perturb * 0.8
                
        y0_int = [x10, y[0]]
        t_eval = np.linspace(0, y[1], 1000)

        sol = solve_ivp(numpy_function_integ, y0=y0_int, t_span=[0, t_eval[-1]],
                            t_eval=t_eval,
                            rtol=1e-5, atol=1e-8)
        
        x1max_pred.append(np.max(sol.y[0,:]))
        x2max_pred.append(np.max(sol.y[1,:]))
        x1min_pred.append(np.min(sol.y[0,:]))
        x2min_pred.append(np.min(sol.y[1,:]))
                
    else:
        if switch is not False:
            switch = False
            unstable_ss.append(np.array([np.nan, np.nan]))
            unstable_B.append(np.nan)
        stable_ss.append(root)
        stable_B.append(B)

        x1max_pred.append(root[0])
        x2max_pred.append(root[1])
        x1min_pred.append(root[0])
        x2min_pred.append(root[1])

    
    roots.append(root)
roots = np.array(roots)


unstable_ss_pred = np.array(unstable_ss)
unstable_B_pred = np.array(unstable_B)
stable_ss_pred = np.array(stable_ss)
stable_B_pred = np.array(stable_B)

In [ ]:
plt.figure()
#     plt.plot(Da_arr,roots[:,0])
if stable_B_pred.size > 0:
    plt.plot(stable_B_pred,stable_ss_pred[...,0],'b')
if unstable_B_pred.size > 0:
    plt.plot(unstable_B_pred,unstable_ss_pred[...,0],'b--')
plt.plot(B_arr, x1min_pred, 'b')
plt.plot(B_arr, x1max_pred, 'b')
plt.xlabel(r'B', fontsize=24)
plt.ylabel(r'$x_1$', fontsize=24)

plt.figure()
#     plt.plot(Da_arr,roots[:,1])
if stable_B_pred.size > 0:
    plt.plot(stable_B_pred,stable_ss_pred[...,1],'b')
if unstable_B_pred.size > 0:
    plt.plot(unstable_B_pred,unstable_ss_pred[...,1],'b--')
plt.plot(B_arr, x2min_pred, 'b')
plt.plot(B_arr, x2max_pred, 'b')
plt.xlabel(r'B', fontsize=24)
plt.ylabel(r'$x_2$', fontsize=24)

In [ ]:
from scipy.optimize import fsolve

def ODE_Bifurc(y, func, Da, x10):
    x2, T = y
    # pars = get_pars(Da)
    
#     event = ODE_Event
#     event.terminal = True
    
    y0 = [x10, x2]

    
    sol = solve_ivp(func, y0=y0, t_span=[0, 0.1],
#                     args=pars,
                    rtol=1e-5, atol=1e-8, dense_output=True)#, events=(event,))#, dense_output=True)
    
    y_init = sol.y[:,-1]
    
    sol = solve_ivp(func, y0=y_init, t_span=[0.1, np.max((T,0.1))],
#                     args=pars,
                    rtol=1e-5, atol=1e-8, dense_output=True)#, events=(event,))#, dense_output=True)
    
    T_out = sol.t[-1]
    x1_out = sol.y[0,-1]
    x2_out = sol.y[1,-1]

    return (x10-x1_out), (x2-x2_out)

##############################################################################################33

B_arr = np.linspace(10,15,100)
Da = 0.3

stable_ss = []
stable_B = []

unstable_ss = []
unstable_B = []

x1max_true = []
x1min_true = []

x2max_true = []
x2min_true = []

roots = []
switch = False
root = [0.5, 1]
for i in range(len(B_arr)):
    print(i)
    B = B_arr[i]
    beta = 3
    
    # pars = get_pars(Da)
    # def true_ode(t, x, Da, B, beta):
    #     x1, x2 = x
    #     dx1dt = -x1 + Da * (1-x1) * np.exp(x2)
    #     dx2dt = -x2 + B * Da * (1-x1) * np.exp(x2) - beta * x2
    #     return np.array([dx1dt, dx2dt])

    def torch_function(x):
        x1, x2 = x
        dx1dt = -x1 + Da * (1-x1) * torch.exp(x2)
        dx2dt = -x2 + B * Da * (1-x1) * torch.exp(x2) - beta * x2
        return torch.stack((dx1dt, dx2dt), dim=-1)

    def numpy_function(x):
        x1, x2 = x
        dx1dt = -x1 + Da * (1-x1) * np.exp(x2)
        dx2dt = -x2 + B * Da * (1-x1) * np.exp(x2) - beta * x2
        return np.stack((dx1dt, dx2dt), axis=-1)

    def numpy_function_integ(t, x):
        x1, x2 = x
        dx1dt = -x1 + Da * (1-x1) * np.exp(x2)
        dx2dt = -x2 + B * Da * (1-x1) * np.exp(x2) - beta * x2
        return np.stack((dx1dt, dx2dt), axis=-1)

    
    root = fsolve(numpy_function, root)
    J = torch.autograd.functional.jacobian(torch_function, torch.from_numpy(root), strict=True)
    w, v = np.linalg.eig(J)

    if np.any(w.real>0):
        if switch is not True:
            switch = True
            stable_ss.append(np.array([np.nan, np.nan]))
            stable_B.append(np.nan)
            
        unstable_ss.append(root)
        unstable_B.append(B)

        solved = False
        perturb = 0.1
        
        y0 = [root[1], 2]
        while not solved:
            x10 = root[0] + perturb
            

            y, infodict, ier, mesg = fsolve(ODE_Bifurc, y0, args=(numpy_function_integ, Da, x10), full_output=True)
            
            if ier == 1:
                solved = True
                y0=y
            else:
                perturb = perturb * 0.8
                
        y0_int = [x10, y[0]]
        t_eval = np.linspace(0, y[1], 1000)

        sol = solve_ivp(numpy_function_integ, y0=y0_int, t_span=[0, t_eval[-1]],
                            t_eval=t_eval,
                            rtol=1e-5, atol=1e-8)
        
        x1max_true.append(np.max(sol.y[0,:]))
        x2max_true.append(np.max(sol.y[1,:]))
        x1min_true.append(np.min(sol.y[0,:]))
        x2min_true.append(np.min(sol.y[1,:]))
                
    else:
        if switch is not False:
            switch = False
            unstable_ss.append(np.array([np.nan, np.nan]))
            unstable_B.append(np.nan)
        stable_ss.append(root)
        stable_B.append(B)

        x1max_true.append(root[0])
        x2max_true.append(root[1])
        x1min_true.append(root[0])
        x2min_true.append(root[1])

    
    roots.append(root)
roots = np.array(roots)


unstable_ss_true = np.array(unstable_ss)
unstable_B_true = np.array(unstable_B)
stable_ss_true = np.array(stable_ss)
stable_B_true = np.array(stable_B)

In [ ]:
# print(root)
# print(x2min_true)
# print(x2max_true)
# print(ODE_Bifurc(y, numpy_function_integ, Da, x10))

In [ ]:
plt.figure()
if stable_B_true.size > 0:
    plt.plot(stable_B_true,stable_ss_true[...,0],'k')
if unstable_B_true.size > 0:
    plt.plot(unstable_B_true,unstable_ss_true[...,0],'k--')
plt.plot(B_arr, x1min_true, 'k')
plt.plot(B_arr, x1max_true, 'k')

if stable_B_pred.size > 0:
    plt.plot(stable_B_pred,stable_ss_pred[...,0],'b')
if unstable_B_pred.size > 0:
    plt.plot(unstable_B_pred,unstable_ss_pred[...,0],'b--')
plt.plot(B_arr, x1min_pred, 'b')
plt.plot(B_arr, x1max_pred, 'b')

plt.xlabel(r'B', fontsize=24)
plt.ylabel(r'$x_1$', fontsize=24)

plt.figure()
if stable_B_true.size > 0:
    plt.plot(stable_B_true,stable_ss_true[...,1],'k')
if unstable_B_true.size > 0:
    plt.plot(unstable_B_true,unstable_ss_true[...,1],'k--')
plt.plot(B_arr, x2min_true, 'k')
plt.plot(B_arr, x2max_true, 'k')

if stable_B_pred.size > 0:
    plt.plot(stable_B_pred,stable_ss_pred[...,1],'b')
if unstable_B_pred.size > 0:
    plt.plot(unstable_B_pred,unstable_ss_pred[...,1],'b--')
plt.plot(B_arr, x2min_pred, 'b')
plt.plot(B_arr, x2max_pred, 'b')

plt.xlabel(r'B', fontsize=24)
plt.ylabel(r'$x_2$', fontsize=24)



In [ ]:
from scipy.optimize import fsolve

def ODE_Bifurc(y, func, Da, x10):
    x2, T = y
    # pars = get_pars(Da)
    
#     event = ODE_Event
#     event.terminal = True
    
    y0 = [x10, x2]

    
    sol = solve_ivp(func, y0=y0, t_span=[0, 0.1],
#                     args=pars,
                    rtol=1e-5, atol=1e-8, dense_output=True)#, events=(event,))#, dense_output=True)
    
    y_init = sol.y[:,-1]
    
    sol = solve_ivp(func, y0=y_init, t_span=[0.1, np.max((T,0.1))],
#                     args=pars,
                    rtol=1e-5, atol=1e-8, dense_output=True)#, events=(event,))#, dense_output=True)
    
    T_out = sol.t[-1]
    x1_out = sol.y[0,-1]
    x2_out = sol.y[1,-1]

    return (x10-x1_out), (x2-x2_out)

##############################################################################################33

beta_arr = np.linspace(1,5,100)
Da = 0.3

stable_ss = []
stable_beta = []

unstable_ss = []
unstable_beta = []

x1max_true = []
x1min_true = []

x2max_true = []
x2min_true = []

roots = []
switch = False
root = [0.5, 1]
for i in range(len(beta_arr)):
    # print(i)
    beta = beta_arr[i]
    B = 11
    
    # pars = get_pars(Da)
    # def true_ode(t, x, Da, B, beta):
    #     x1, x2 = x
    #     dx1dt = -x1 + Da * (1-x1) * np.exp(x2)
    #     dx2dt = -x2 + B * Da * (1-x1) * np.exp(x2) - beta * x2
    #     return np.array([dx1dt, dx2dt])

    def torch_function(x):
        x1, x2 = x
        dx1dt = -x1 + Da * (1-x1) * torch.exp(x2)
        dx2dt = -x2 + B * Da * (1-x1) * torch.exp(x2) - beta * x2
        return torch.stack((dx1dt, dx2dt), dim=-1)

    def numpy_function(x):
        x1, x2 = x
        dx1dt = -x1 + Da * (1-x1) * np.exp(x2)
        dx2dt = -x2 + B * Da * (1-x1) * np.exp(x2) - beta * x2
        return np.stack((dx1dt, dx2dt), axis=-1)

    def numpy_function_integ(t, x):
        x1, x2 = x
        dx1dt = -x1 + Da * (1-x1) * np.exp(x2)
        dx2dt = -x2 + B * Da * (1-x1) * np.exp(x2) - beta * x2
        return np.stack((dx1dt, dx2dt), axis=-1)

    
    root = fsolve(numpy_function, root)
    J = torch.autograd.functional.jacobian(torch_function, torch.from_numpy(root), strict=True)
    w, v = np.linalg.eig(J)

    if np.any(w.real>0):
        if switch is not True:
            switch = True
            stable_ss.append(np.array([np.nan, np.nan]))
            stable_beta.append(np.nan)
            
        unstable_ss.append(root)
        unstable_beta.append(beta)

        solved = False
        perturb = 0.1
        
        y0 = [root[1], 2]
        while not solved:
            x10 = root[0] + perturb
            

            y, infodict, ier, mesg = fsolve(ODE_Bifurc, y0, args=(numpy_function_integ, Da, x10), full_output=True)
            
            if ier == 1:
                solved = True
                y0=y
            else:
                perturb = perturb * 0.8
                
        y0_int = [x10, y[0]]
        t_eval = np.linspace(0, y[1], 1000)

        sol = solve_ivp(numpy_function_integ, y0=y0_int, t_span=[0, t_eval[-1]],
                            t_eval=t_eval,
                            rtol=1e-5, atol=1e-8)
        
        x1max_true.append(np.max(sol.y[0,:]))
        x2max_true.append(np.max(sol.y[1,:]))
        x1min_true.append(np.min(sol.y[0,:]))
        x2min_true.append(np.min(sol.y[1,:]))
                
    else:
        if switch is not False:
            switch = False
            unstable_ss.append(np.array([np.nan, np.nan]))
            unstable_beta.append(np.nan)
        stable_ss.append(root)
        stable_beta.append(beta)

        x1max_true.append(root[0])
        x2max_true.append(root[1])
        x1min_true.append(root[0])
        x2min_true.append(root[1])

    
    roots.append(root)
roots = np.array(roots)


unstable_ss_true = np.array(unstable_ss)
unstable_beta_true = np.array(unstable_beta)
stable_ss_true = np.array(stable_ss)
stable_beta_true = np.array(stable_beta)

In [ ]:
from scipy.optimize import fsolve

def ODE_Bifurc(y, func, Da, x10):
    x2, T = y
    # pars = get_pars(Da)
    
#     event = ODE_Event
#     event.terminal = True
    
    y0 = [x10, x2]

    
    sol = solve_ivp(func, y0=y0, t_span=[0, 0.1],
#                     args=pars,
                    rtol=1e-5, atol=1e-8, dense_output=True)#, events=(event,))#, dense_output=True)
    
    y_init = sol.y[:,-1]
    
    sol = solve_ivp(func, y0=y_init, t_span=[0.1, T],
#                     args=pars,
                    rtol=1e-5, atol=1e-8, dense_output=True)#, events=(event,))#, dense_output=True)
    
    T_out = sol.t[-1]
    x1_out = sol.y[0,-1]
    x2_out = sol.y[1,-1]

    return (x10-x1_out), (x2-x2_out)

##############################################################################################33

beta_arr = np.linspace(1,5,100)
Da = 0.3

stable_ss = []
stable_beta = []

unstable_ss = []
unstable_beta = []

x1max_pred = []
x1min_pred = []

x2max_pred = []
x2min_pred = []

roots = []
switch = False
root = [0.5, 1]
for i in range(len(beta_arr)):
    print(i)
    beta = beta_arr[i]
    B = 11
    
    # pars = get_pars(Da)

    def torch_function(x):
        g = network.raw_output(x.unsqueeze(0).to(network.device),
                                                torch.tensor([Da]).unsqueeze(0).to(network.device))
        x1, x2 = x
        dx1dt = -x1 + g
        dx2dt = -x2 + B * g - beta * x2
        return torch.cat((dx1dt, dx2dt), dim=-1)

    def numpy_function(x):
        g = network.raw_output(torch.tensor(x).unsqueeze(0).to(network.device),
                                                   torch.tensor([Da]).unsqueeze(0).to(network.device))
        x1, x2 = x
        dx1dt = -x1 + g
        dx2dt = -x2 + B * g - beta * x2
        return torch.cat((dx1dt, dx2dt), dim=-1).detach().cpu().squeeze().numpy()

    def numpy_function_integ(t, x):
        g = network.raw_output(torch.tensor(x).unsqueeze(0).to(network.device),
                                                   torch.tensor([Da]).unsqueeze(0).to(network.device))
        x1, x2 = x
        dx1dt = -x1 + g
        dx2dt = -x2 + B * g - beta * x2
        return torch.cat((dx1dt, dx2dt), dim=-1).detach().cpu().squeeze().numpy()

    
    root = fsolve(numpy_function, root)
    J = torch.autograd.functional.jacobian(torch_function, torch.from_numpy(root), strict=True)
    w, v = np.linalg.eig(J)

    if np.any(w.real>0):
        if switch is not True:
            switch = True
            stable_ss.append(np.array([np.nan, np.nan]))
            stable_beta.append(np.nan)
            
        unstable_ss.append(root)
        unstable_beta.append(beta)

        solved = False
        perturb = 0.1
        
        y0 = [root[1], 2]
        while not solved:
            x10 = root[0] + perturb
            

            y, infodict, ier, mesg = fsolve(ODE_Bifurc, y0, args=(numpy_function_integ, Da, x10), full_output=True)
            
            if ier == 1:
                solved = True
                y0=y
            else:
                perturb = perturb * 0.8
                
        y0_int = [x10, y[0]]
        t_eval = np.linspace(0, y[1], 1000)

        sol = solve_ivp(numpy_function_integ, y0=y0_int, t_span=[0, t_eval[-1]],
                            t_eval=t_eval,
                            rtol=1e-5, atol=1e-8)
        
        x1max_pred.append(np.max(sol.y[0,:]))
        x2max_pred.append(np.max(sol.y[1,:]))
        x1min_pred.append(np.min(sol.y[0,:]))
        x2min_pred.append(np.min(sol.y[1,:]))
                
    else:
        if switch is not False:
            switch = False
            unstable_ss.append(np.array([np.nan, np.nan]))
            unstable_beta.append(np.nan)
        stable_ss.append(root)
        stable_beta.append(beta)

        x1max_pred.append(root[0])
        x2max_pred.append(root[1])
        x1min_pred.append(root[0])
        x2min_pred.append(root[1])

    
    roots.append(root)
roots = np.array(roots)


unstable_ss_pred = np.array(unstable_ss)
unstable_beta_pred = np.array(unstable_beta)
stable_ss_pred = np.array(stable_ss)
stable_beta_pred = np.array(stable_beta)

In [ ]:
plt.figure()
if stable_beta_true.size > 0:
    plt.plot(stable_beta_true,stable_ss_true[...,0],'k')
if unstable_beta_true.size > 0:
    plt.plot(unstable_beta_true,unstable_ss_true[...,0],'k--')
plt.plot(beta_arr, x1min_true, 'k')
plt.plot(beta_arr, x1max_true, 'k')

if stable_beta_pred.size > 0:
    plt.plot(stable_beta_pred,stable_ss_pred[...,0],'b')
if unstable_beta_pred.size > 0:
    plt.plot(unstable_beta_pred,unstable_ss_pred[...,0],'b--')
plt.plot(beta_arr, x1min_pred, 'b')
plt.plot(beta_arr, x1max_pred, 'b')

plt.xlabel(r'$\beta$', fontsize=24)
plt.ylabel(r'$x_1$', fontsize=24)

plt.figure()
if stable_beta_true.size > 0:
    plt.plot(stable_beta_true,stable_ss_true[...,1],'k')
if unstable_beta_true.size > 0:
    plt.plot(unstable_beta_true,unstable_ss_true[...,1],'k--')
plt.plot(beta_arr, x2min_true, 'k')
plt.plot(beta_arr, x2max_true, 'k')

if stable_beta_pred.size > 0:
    plt.plot(stable_beta_pred,stable_ss_pred[...,1],'b')
if unstable_beta_pred.size > 0:
    plt.plot(unstable_beta_pred,unstable_ss_pred[...,1],'b--')
plt.plot(beta_arr, x2min_pred, 'b')
plt.plot(beta_arr, x2max_pred, 'b')

plt.xlabel(r'$\beta$', fontsize=24)
plt.ylabel(r'$x_2$', fontsize=24)

